In [1]:
#load dependencies 
import torch
import torch.nn as nn
import torch.optim as optim
import torchvision
import torchvision.transforms as transforms
from torch.utils.tensorboard import SummaryWriter

writer = SummaryWriter(log_dir="runs/cifar10_experiment_task1_3_e50")

# Task 1.1

In [2]:
# Device (CPU/GPU)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Data preprocessing
transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.5, 0.5, 0.5),
                         (0.5, 0.5, 0.5))
])

# Load CIFAR-10 dataset
trainset = torchvision.datasets.CIFAR10(
    root='./data', train=True, download=True, transform=transform)

trainloader = torch.utils.data.DataLoader(
    trainset, batch_size=64, shuffle=True)

testset = torchvision.datasets.CIFAR10(
    root='./data', train=False, download=True, transform=transform)

testloader = torch.utils.data.DataLoader(
    testset, batch_size=64, shuffle=False)

In [3]:
class SimpleCNN(nn.Module):
    def __init__(self):
        super(SimpleCNN, self).__init__()

        self.conv1 = nn.Conv2d(3, 32, kernel_size=3, padding=1)
        self.conv2 = nn.Conv2d(32, 64, kernel_size=3, padding=1)

        self.pool = nn.MaxPool2d(2, 2)

        self.fc1 = nn.Linear(64 * 8 * 8, 256)
        self.fc2 = nn.Linear(256, 10)

        self.activation = nn.LeakyReLU()

    def forward(self, x):
        x = self.pool(self.activation(self.conv1(x)))
        x = self.pool(self.activation(self.conv2(x)))

        x = x.view(-1, 64 * 8 * 8)

        x = self.activation(self.fc1(x))
        x = self.fc2(x)

        return x

# Initialize model
model = SimpleCNN().to(device)

## Task 1.3

In [4]:
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.0001)

epochs = 50

for epoch in range(epochs):
    running_loss = 0.0

    for inputs, labels in trainloader:
        inputs, labels = inputs.to(device), labels.to(device)

        optimizer.zero_grad()

        outputs = model(inputs)
        loss = criterion(outputs, labels)

        loss.backward()
        optimizer.step()

        running_loss += loss.item()

    avg_loss = running_loss / len(trainloader)
    print(f"Epoch {epoch+1}, Loss: {avg_loss:.4f}")

    writer.add_scalar("Loss/train", avg_loss, epoch)

Epoch 1, Loss: 1.6566
Epoch 2, Loss: 1.3384
Epoch 3, Loss: 1.2314
Epoch 4, Loss: 1.1564
Epoch 5, Loss: 1.0928
Epoch 6, Loss: 1.0388
Epoch 7, Loss: 0.9936
Epoch 8, Loss: 0.9510
Epoch 9, Loss: 0.9144
Epoch 10, Loss: 0.8773
Epoch 11, Loss: 0.8448
Epoch 12, Loss: 0.8117
Epoch 13, Loss: 0.7809
Epoch 14, Loss: 0.7555
Epoch 15, Loss: 0.7250
Epoch 16, Loss: 0.6949
Epoch 17, Loss: 0.6661
Epoch 18, Loss: 0.6425
Epoch 19, Loss: 0.6138
Epoch 20, Loss: 0.5863
Epoch 21, Loss: 0.5615
Epoch 22, Loss: 0.5325
Epoch 23, Loss: 0.5044
Epoch 24, Loss: 0.4862
Epoch 25, Loss: 0.4561
Epoch 26, Loss: 0.4291
Epoch 27, Loss: 0.4047
Epoch 28, Loss: 0.3795
Epoch 29, Loss: 0.3547
Epoch 30, Loss: 0.3277
Epoch 31, Loss: 0.3053
Epoch 32, Loss: 0.2840
Epoch 33, Loss: 0.2625
Epoch 34, Loss: 0.2411
Epoch 35, Loss: 0.2186
Epoch 36, Loss: 0.1981
Epoch 37, Loss: 0.1802
Epoch 38, Loss: 0.1609
Epoch 39, Loss: 0.1445
Epoch 40, Loss: 0.1347
Epoch 41, Loss: 0.1169
Epoch 42, Loss: 0.1045
Epoch 43, Loss: 0.0924
Epoch 44, Loss: 0.08

In [5]:
correct = 0
total = 0

model.eval()
with torch.no_grad():
    for images, labels in testloader:
        images, labels = images.to(device), labels.to(device)

        outputs = model(images)
        _, predicted = torch.max(outputs, 1)

        total += labels.size(0)
        correct += (predicted == labels).sum().item()

accuracy = 100 * correct / total
print(f"Test Accuracy: {accuracy:.2f}%")

writer.add_scalar("Accuracy/test", accuracy)

writer.close()

Test Accuracy: 69.26%
